In [1]:
import pandas as pd

#importing all the files. remember they are in the ravenstack folder
accounts = pd.read_csv("ravenstack_accounts.csv")
churn = pd.read_csv("ravenstack_churn_events.csv")
subs = pd.read_csv("ravenstack_subscriptions.csv")
tickets = pd.read_csv("ravenstack_support_tickets.csv")
usage = pd.read_csv("ravenstack_feature_usage.csv")

print("All 5 files have been loaded successfully! Now, let's get to cleaning!")

All 5 files have been loaded successfully! Now, let's get to cleaning!


In [4]:
#1. summarising the support tickets

ticket_summary = tickets.groupby('account_id').agg(
    total_support_tickets=('ticket_id', 'count'),
    avg_resolution_time_hours=('resolution_time_hours', 'mean'),
    avg_satisfaction_score=('satisfaction_score', 'mean'),
    total_escalated_tickets=('escalation_flag', 'sum')
).reset_index()

#2. summarising feature usage
#mapping first to accounts
usage_mapped = pd.merge(usage, subs[['subscription_id', 'account_id']], on= 'subscription_id', how='left')

#grouping usage logs by account
usage_summary = usage_mapped.groupby('account_id').agg(
    total_feature_clicks=('usage_count', 'sum'),
    total_system_errors=('error_count', sum)
).reset_index()

#MASTER table
#first, MASTER accounts
master_df = pd.merge(accounts, ticket_summary, on='account_id', how='left')
master_df = pd.merge(master_df, usage_summary, on='account_id', how='left')

#adding churn
master_df = pd.merge(master_df, churn[['account_id', 'reason_code', 'feedback_text']], on='account_id', how='left')

#data cleaning
fill_values = {
    'total_support_tickets': 0,
    'avg_resolution_time_hours': 0,
    'total_escalated_tickets': 0,
    'total_feature_clicks': 0,
    'reason_code': 'Active',
    'feedback_text': 'None',
}
master_df.fillna(value=fill_values, inplace=True)
master_df['avg_satisfaction_score'].fillna(5.0, inplace=True)

#THE TABLE!
print("Merging is a great success! Herewith are the first 3 rows of the new MASTER table:")
display(master_df.head(3))

Merging is a great success! Herewith are the first 3 rows of the new MASTER table:


C:\Users\junio\AppData\Local\Temp\ipykernel_31076\1167972672.py:38: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  master_df['avg_satisfaction_score'].fillna(5.0, inplace=True)


,account_id,account_name,industry,country,signup_date,referral_source,plan_tier,seats,is_trial,churn_flag,total_support_tickets,avg_resolution_time_hours,avg_satisfaction_score,total_escalated_tickets,total_feature_clicks,total_system_errors,reason_code,feedback_text
0,A-2e4581,Company_0,EdTech,US,2024-10-16,partner,Basic,9,False,False,2.0,23.0,3.0,0.0,535,38,budget,switched to competitor
1,A-2e4581,Company_0,EdTech,US,2024-10-16,partner,Basic,9,False,False,2.0,23.0,3.0,0.0,535,38,competitor,None
2,A-43a9e3,Company_1,FinTech,IN,2023-08-17,other,Basic,18,False,True,3.0,38.0,4.0,0.0,355,14,Active,None


In [5]:
master_df.to_csv ("ravenstack_master_table_clean.csv", index=False)

print("File saved successfully!")

File saved successfully!
